# Capacity Metrics Ingestion: Timepoint-Level Detail

Pulls interactive and background operation detail from the Fabric Capacity Metrics
semantic model at the timepoint grain and persists it to a lakehouse Delta table.

This notebook focuses on **ingestion only**. Presentation and costing come later.

**Data source:** The `Timepoint Interactive Detail` and `Timepoint Background Detail`
tables inside the Capacity Metrics app's semantic model, queried via `sempy.fabric.evaluate_dax`.

**Sampling strategy:** One timepoint per hour (on the hour, 30-second mark).
Interactive operations are smoothed over 5 to 64 minutes, so hourly sampling captures
most operations but is not perfectly exhaustive. If you need every operation, drop the
interval to every 30 seconds (2,880 calls per day), but that is heavy on the capacity.
Background operations are smoothed over 24 hours, so a single daily timepoint is
technically sufficient. We still pull hourly for both to keep the pattern uniform and
to capture the timepoint-level CU distribution across the day.

**Architecture:** Follows the same patterns as `nb_run_activities` and `nb_run_datasets`.
Uses `nb_udfs` for shared utilities, `ThreadPoolExecutor` for parallel DAX calls,
`union_batches` for combining results, and `udf_UpsertDimension` for Delta persistence.

**Prerequisites**
- Run in a Fabric notebook attached to the target lakehouse.
- Read + Build permission on the Fabric Capacity Metrics semantic model.
- XMLA endpoint enabled on the capacity hosting the Metrics workspace.
- `nb_udfs` notebook available in the same workspace.


## Pull in UDFs

In [25]:
%run nb_udfs

StatementMeta(, ae6c604e-66c6-44da-9da4-3258208d3d6b, 59, Finished, Available, Finished, True)

## Configuration

In [26]:
# -----------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------

# Workspace and lakehouse for persisting the output.
# Uses URL-encoded name if the workspace contains special characters.
#workspace = 'BI%20%26%20Analytics%20Administration'
workspace = 'Fabric%20of%20Middle-Earth'
lakehouse = 'lh_capacity_management'

# Delta table names for the two operation types.
interactive_table = 'factCapacityMetricsInteractive'
background_table = 'factCapacityMetricsBackground'

# Capacity Metrics semantic model location.
# These are specific to your Metrics app installation and rarely change.
METRICS_WORKSPACE_ID = 'f3bff753-0af3-4e50-af24-891b4d5a63c4'
METRICS_DATASET_ID   = '00f96c02-90a8-4389-b5ed-669ae6f741e8'

# How many days back to pull. 1 = yesterday only.
# The Metrics app retains 14 days by default (30 with preview feature).
date_offset = 2

# Number of parallel DAX calls.
# Keep this moderate to avoid throttling the Metrics semantic model.
max_workers = 6


StatementMeta(, ae6c604e-66c6-44da-9da4-3258208d3d6b, 60, Finished, Available, Finished, False)

In [27]:
# Get all active Fabric capacities from the admin API.
response = _base_api(
    request="/v1.0/myorg/admin/capacities",
    method="get"
)
df_capacities = pd.json_normalize(response.json()['value'])
df_capacities = df_capacities.drop(columns=["admins", "users"], errors="ignore")

# Filter to active capacities only.
active_capacities = df_capacities[df_capacities['state'] == 'Active']['id'].tolist()
print(f"Found {len(active_capacities)} active capacities:")
for cap_id in active_capacities:
    match = df_capacities[df_capacities['id'] == cap_id]
    name = match['displayName'].values[0] if 'displayName' in match.columns else 'unknown'
    print(f"  {cap_id} - {name}")

StatementMeta(, ae6c604e-66c6-44da-9da4-3258208d3d6b, 61, Finished, Available, Finished, False)

Found 2 active capacities:
  bbece78e-f9e4-489f-814a-874b18c151d7 - dragonvault
  08C8F924-B357-4B8E-8CFD-F6FED5ED37F6 - Premium Per User - Reserved


## DAX Query Builders

In [28]:
def build_interactive_dax(timepoint_dt, capacity_id):
    """
    Build a DAX query that returns all interactive operations visible at a given timepoint.
    The timepoint must be on a 30-second boundary (seconds = 0 or 30).
    Returns every column from the app's interactive operations table except % of Base Capacity.
    """
    tp = (
        f"DATE({timepoint_dt.year}, {timepoint_dt.month}, {timepoint_dt.day}) "
        f"+ TIME({timepoint_dt.hour}, {timepoint_dt.minute}, {timepoint_dt.second})"
    )
    return f"""
DEFINE
    MPARAMETER 'TimePoint' = ({tp})
    MPARAMETER 'CapacitiesList' = {{ "{capacity_id}" }}

    VAR __ops =
        SUMMARIZECOLUMNS(
            'Timepoint Interactive Detail'[Operation start time],
            'Timepoint Interactive Detail'[Operation end time],
            'Timepoint Interactive Detail'[Status],
            'Timepoint Interactive Detail'[Operation],
            'Timepoint Interactive Detail'[User],
            'Timepoint Interactive Detail'[Operation Id],
            'Timepoint Interactive Detail'[Billing type],
            'Items'[Workspace Id],
            'Items'[Workspace name],
            'Items'[Item kind],
            'Items'[Item Id],
            'Items'[Item name],
            "Duration_s",     SUM('Timepoint Interactive Detail'[Duration (s)]),
            "Timepoint_CU_s", SUM('Timepoint Interactive Detail'[Timepoint CU (s)]),
            "Total_CU_s",     SUM('Timepoint Interactive Detail'[Total CU (s)]),
            "Throttling_s",   SUM('Timepoint Interactive Detail'[Throttling (s)])
        )

EVALUATE __ops
ORDER BY
    'Timepoint Interactive Detail'[Operation start time] DESC
"""


def build_background_dax(timepoint_dt, capacity_id):
    """
    Build a DAX query that returns all background operations visible at a given timepoint.
    Same structure as interactive but targets the background detail table.
    """
    tp = (
        f"DATE({timepoint_dt.year}, {timepoint_dt.month}, {timepoint_dt.day}) "
        f"+ TIME({timepoint_dt.hour}, {timepoint_dt.minute}, {timepoint_dt.second})"
    )
    return f"""
DEFINE
    MPARAMETER 'TimePoint' = ({tp})
    MPARAMETER 'CapacitiesList' = {{ "{capacity_id}" }}

    VAR __ops =
        SUMMARIZECOLUMNS(
            'Timepoint Background Detail'[Operation start time],
            'Timepoint Background Detail'[Operation end time],
            'Timepoint Background Detail'[Status],
            'Timepoint Background Detail'[Operation],
            'Timepoint Background Detail'[User],
            'Timepoint Background Detail'[Operation Id],
            'Timepoint Background Detail'[Billing type],
            'Items'[Workspace Id],
            'Items'[Workspace name],
            'Items'[Item kind],
            'Items'[Item Id],
            'Items'[Item name],
            "Duration_s",     SUM('Timepoint Background Detail'[Duration (s)]),
            "Timepoint_CU_s", SUM('Timepoint Background Detail'[Timepoint CU (s)]),
            "Total_CU_s",     SUM('Timepoint Background Detail'[Total CU (s)]),
            "Throttling_s",   SUM('Timepoint Background Detail'[Throttling (s)])
        )

EVALUATE __ops
ORDER BY
    'Timepoint Background Detail'[Operation start time] DESC
"""


StatementMeta(, ae6c604e-66c6-44da-9da4-3258208d3d6b, 62, Finished, Available, Finished, False)

## DAX Execution Helpers

In [29]:
def run_dax(dax_string):
    """
    Execute a DAX query against the Capacity Metrics semantic model.
    Returns a pandas DataFrame.
    """
    return fabric.evaluate_dax(
        METRICS_DATASET_ID,
        dax_string,
        workspace=METRICS_WORKSPACE_ID
    )


def generate_hourly_timepoints(date_str):
    """
    Generate one timepoint per hour for a given date string (YYYY-MM-DD).
    Each timepoint lands on the 30-second mark of the top of the hour
    (e.g., 00:00:30, 01:00:30, ..., 23:00:30) to align with the Capacity
    Metrics 30-second timepoint grid.
    Returns a list of datetime objects.
    """
    base = dt.strptime(date_str, '%Y-%m-%d')
    return [
        base + timedelta(hours=h, seconds=30)
        for h in range(24)
    ]


StatementMeta(, ae6c604e-66c6-44da-9da4-3258208d3d6b, 63, Finished, Available, Finished, False)

## Schema Probe

Run a single timepoint query for each operation type to confirm the table and
column names match your version of the Capacity Metrics app. If either call
errors with a table-not-found or column-not-found message, the semantic model
may use older naming conventions. Check the error and adjust the DAX builders.

In [30]:
# Pick one timepoint from yesterday to test.
target_date = (dt.utcnow() - timedelta(days=date_offset)).strftime('%Y-%m-%d')
test_timepoints = generate_hourly_timepoints(target_date)
test_tp = test_timepoints[12]  # noon-ish, likely to have data

print(f"Probing timepoint: {test_tp}")
print(f"Target date: {target_date}")
print()

# Test interactive
try:
    df_probe_int = run_dax(build_interactive_dax(test_tp, CAPACITY_ID))
    print(f"Interactive probe: {len(df_probe_int)} rows returned")
    if not df_probe_int.empty:
        print(f"Columns: {list(df_probe_int.columns)}")
        display(df_probe_int.head(3))
except Exception as e:
    print(f"Interactive probe FAILED: {str(e)[:200]}")

print()

# Test background
try:
    df_probe_bg = run_dax(build_background_dax(test_tp, CAPACITY_ID))
    print(f"Background probe: {len(df_probe_bg)} rows returned")
    if not df_probe_bg.empty:
        print(f"Columns: {list(df_probe_bg.columns)}")
        display(df_probe_bg.head(3))
except Exception as e:
    print(f"Background probe FAILED: {str(e)[:200]}")


StatementMeta(, ae6c604e-66c6-44da-9da4-3258208d3d6b, 64, Finished, Available, Finished, False)

Probing timepoint: 2026-08-24 12:00:30
Target date: 2026-08-24

Interactive probe FAILED: name 'CAPACITY_ID' is not defined

Background probe FAILED: name 'CAPACITY_ID' is not defined


## Column Cleanup

The DAX output returns fully qualified column names like
`Timepoint Background Detail[Operation start time]` and measure names like
`[Duration_s]`. This function strips them down to clean snake_case names
suitable for Delta persistence.

In [31]:
def clean_columns(df, operation_type, sampled_timepoint, capacity_id):
    """
    Rename the raw DAX output columns to clean snake_case names,
    add metadata columns, and return a pandas DataFrame.
    """
    rename_map = {}
    for col_name in df.columns:
        # Strip table prefix: 'Table Name[Column]' -> 'Column'
        # Strip bare brackets: '[Measure]' -> 'Measure'
        clean = col_name
        if '[' in clean and ']' in clean:
            clean = clean[clean.index('[') + 1 : clean.index(']')]
        # Normalize to snake_case
        clean = (
            clean
            .strip()
            .replace(' ', '_')
            .replace('(', '')
            .replace(')', '')
            .lower()
        )
        rename_map[col_name] = clean

    df = df.rename(columns=rename_map)

    # Add metadata columns
    df['operation_type'] = operation_type
    df['sampled_timepoint'] = sampled_timepoint
    df['capacity_id'] = capacity_id

    return df

StatementMeta(, ae6c604e-66c6-44da-9da4-3258208d3d6b, 65, Finished, Available, Finished, False)

## Fetch and Thread

Each thread handles one hourly timepoint, calling both the interactive and
background DAX queries. Results are collected into lists and unioned after
all threads complete.

In [32]:
def fetch_timepoint(timepoint_dt, capacity_id):
    """
    Pull both interactive and background operations for a single timepoint and capacity.
    Returns a dict with 'interactive' and 'background' keys.
    Each value is either a Spark DataFrame or None.
    """
    results = {'interactive': None, 'background': None}

    # Interactive
    try:
        df_int = run_dax(build_interactive_dax(timepoint_dt, capacity_id))
        if not df_int.empty:
            df_int = clean_columns(df_int, 'interactive', timepoint_dt, capacity_id)
            results['interactive'] = spark.createDataFrame(df_int)
    except Exception as e:
        print(f"   Interactive failed at {timepoint_dt} cap {capacity_id[:8]}: {str(e)[:120]}")

    # Background
    try:
        df_bg = run_dax(build_background_dax(timepoint_dt, capacity_id))
        if not df_bg.empty:
            df_bg = clean_columns(df_bg, 'background', timepoint_dt, capacity_id)
            results['background'] = spark.createDataFrame(df_bg)
    except Exception as e:
        print(f"   Background failed at {timepoint_dt} cap {capacity_id[:8]}: {str(e)[:120]}")

    return results

StatementMeta(, ae6c604e-66c6-44da-9da4-3258208d3d6b, 66, Finished, Available, Finished, False)

## Execute Pull

In [33]:
# Build the list of hourly timepoints for the target date.
target_date = (dt.utcnow() - timedelta(days=date_offset)).strftime('%Y-%m-%d')
timepoints = generate_hourly_timepoints(target_date)
print(f"Target date: {target_date}")
print(f"Timepoints per capacity: {len(timepoints)}")
print(f"Capacities: {len(active_capacities)}")
print(f"Total DAX calls: ~{len(timepoints) * len(active_capacities) * 2}")
print()

# Collect Spark DataFrames across all capacities.
df_interactive_list = []
df_background_list = []

for cap_id in active_capacities:
    cap_name = df_capacities[df_capacities['id'] == cap_id]['displayName'].values[0] if 'displayName' in df_capacities.columns else cap_id[:8]
    print(f"Pulling capacity: {cap_name} ({cap_id[:8]}...)")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_tp = {
            executor.submit(fetch_timepoint, tp, cap_id): tp for tp in timepoints
        }
        for i, future in enumerate(as_completed(future_to_tp), 1):
            tp = future_to_tp[future]
            try:
                result = future.result()
                if result['interactive'] is not None:
                    df_interactive_list.append(result['interactive'])
                if result['background'] is not None:
                    df_background_list.append(result['background'])
            except Exception as e:
                print(f"  Failed for timepoint {tp}: {e}")

            if i % 6 == 0:
                print(f"  Completed {i}/{len(timepoints)} timepoints...")

    print(f"  Done with {cap_name}.")
    print()

print(f"All capacities complete. Interactive batches: {len(df_interactive_list)}, Background batches: {len(df_background_list)}")

StatementMeta(, ae6c604e-66c6-44da-9da4-3258208d3d6b, 67, Finished, Available, Finished, False)

Target date: 2026-08-24
Timepoints per capacity: 24
Capacities: 2
Total DAX calls: ~96

Pulling capacity: dragonvault (bbece78e...)
  Completed 6/24 timepoints...
  Completed 12/24 timepoints...
  Completed 18/24 timepoints...
  Completed 24/24 timepoints...
  Done with dragonvault.

Pulling capacity: Premium Per User - Reserved (08C8F924...)
   Interactive failed at 2026-08-24 02:00:30 cap 08C8F924: An error occurred when running AdomdCommand. AdomdCommandActivityId: '93e338ef-8a08-4fdf-bec8-a8126554e0e2'

Caused by A
   Interactive failed at 2026-08-24 05:00:30 cap 08C8F924: An error occurred when running AdomdCommand. AdomdCommandActivityId: '0e6c986f-c3cb-4948-95d3-e146d869ce90'

Caused by A
   Interactive failed at 2026-08-24 00:00:30 cap 08C8F924: An error occurred when running AdomdCommand. AdomdCommandActivityId: '34bc9d5a-5a70-456d-bf3d-d6043a486d58'

Caused by A
   Interactive failed at 2026-08-24 03:00:30 cap 08C8F924: An error occurred when running AdomdCommand. AdomdComman

## Union Batches

In [34]:
df_interactive = None
df_background = None

def union_interactive():
    global df_interactive
    if df_interactive_list:
        df_interactive = union_batches(df_interactive_list, batch_size=50)
        print(f"Interactive: {df_interactive.count()} total rows")
    else:
        print("No interactive data returned.")

def union_background():
    global df_background
    if df_background_list:
        df_background = union_batches(df_background_list, batch_size=50)
        print(f"Background: {df_background.count()} total rows")
    else:
        print("No background data returned.")

with ThreadPoolExecutor(max_workers=2) as executor:
    executor.submit(union_interactive)
    executor.submit(union_background)


StatementMeta(, ae6c604e-66c6-44da-9da4-3258208d3d6b, 68, Finished, Available, Finished, False)

Interactive: 2 total rows
Background: 5789 total rows


## Persist to Delta

In [35]:
def load_table(df, table_name, label):
    """
    Delete existing rows for the target date and capacity, then append fresh data.
    Keyed on sampled_timepoint date + capacity_id so re-runs are safe.
    """
    if df is None:
        print(f"Skipping {label} load (no data).")
        return

    path = udf_GetFilePath(workspace, lakehouse, table_name)

    if notebookutils.fs.exists(path):
        target = DeltaTable.forPath(spark, path)
        target.delete(
            (col("capacity_id").isin(active_capacities))
            & (col("sampled_timepoint").cast("date") == target_date)
        )
        print(f"{label}: deleted existing rows for {target_date}")

        df.write.format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .save(path)
    else:
        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .save(path)
        print(f"{label}: initial load")

    print(f"{label}: {df.count()} rows written")

with ThreadPoolExecutor(max_workers=2) as executor:
    executor.submit(load_table, df_interactive, interactive_table, "Interactive")
    executor.submit(load_table, df_background, background_table, "Background")

StatementMeta(, ae6c604e-66c6-44da-9da4-3258208d3d6b, 69, Finished, Available, Finished, False)

Background: initial load
Interactive: initial load
Interactive: 2 rows written
Background: 5789 rows written


In [36]:
# Sync the SQL analytics endpoint so the new tables are queryable.
logs = udf_SyncSqlEndpoint(workspace, lakehouse)
for log in logs:
    print(log)

StatementMeta(, ae6c604e-66c6-44da-9da4-3258208d3d6b, 70, Finished, Available, Finished, False)

Table: factCapacityMetricsBackground   Last Update: 2026-08-26T18:04:52.246946Z  Table Status: Success  Table Errors: None
Table: factCapacityMetricsInteractive   Last Update: 2026-08-26T18:04:52.7162472Z  Table Status: Success  Table Errors: None
Table: factCapacityMetricsBackground   Last Update: 2026-08-26T18:04:52.246946Z  Table Status: Success  Table Errors: None
Table: factCapacityMetricsInteractive   Last Update: 2026-08-26T18:04:52.7162472Z  Table Status: Success  Table Errors: None
